In [1]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# 数据集准备

In [2]:
datas, targets = load_iris(return_X_y = True)

In [3]:
mark = targets < 2
x = datas[mark]
y = targets[mark]

In [4]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

In [5]:
# 影响梯度的点

print("标准化前 - 均值:", x_train.mean(axis=0))
print("标准化前 - 标准差:", x_train.std(axis=0))
# scaler = StandardScaler()           # 1. 创建一个“标准化器”
# x_train = scaler.fit_transform(x_train)  # 2. 计算训练集的 μ 和 σ，然后转换训练集
# x_test = scaler.transform(x_test)        # 3. 用训练集的 μ 和 σ 去转换测试集
print("标准化后 - 均值:", x_train.mean(axis=0))
print("标准化后 - 标准差:", x_train.std(axis=0))

标准化前 - 均值: [5.4725 3.075  2.8975 0.805 ]
标准化前 - 标准差: [0.65018747 0.46757352 1.41120294 0.55156595]
标准化后 - 均值: [5.4725 3.075  2.8975 0.805 ]
标准化后 - 标准差: [0.65018747 0.46757352 1.41120294 0.55156595]


In [6]:
x_train = x_train.T
y_train = y_train.reshape(1, -1)
x_test = x_test.T
y_test = y_test.reshape(1, -1)

In [30]:
print(f"输入样本的特征维度:{x_train.shape[0]}")
print(f"输入样本的样本数量:{x_train.shape[1]}")

输入样本的特征维度:4
输入样本的样本数量:80


# 参数初始化

In [11]:
D = 4
H = 12
O = 1
# 影响梯度的点
# np.random.seed(42)
W1 = np.random.randn(H, D) * 0.01
b1 = np.zeros((H, 1))
W2 = np.random.randn(O, H) * 0.01
b2 = np.zeros((O, 1))

W1_alt = np.random.randn(H, D)
b1 = np.zeros((H, 1))
W2_alt = np.random.randn(O, H)
b2 = np.zeros((O, 1))

In [32]:
print("W1:", W1.shape)   # (10, 4)
print("b1:", b1.shape)   # (10, 1)
print("W2:", W2.shape)   # (1, 10)
print("b2:", b2.shape)   # (1, 1)

W1: (12, 4)
b1: (12, 1)
W2: (1, 12)
b2: (1, 1)


In [7]:
x = x_train
y = y_train

In [9]:
def forward(x, W1, b1, W2, b2):
    z1 = np.dot(W1, x) + b1

    a1 = np.maximum(0, z1)
    
    z2 = np.dot(W2, a1) + b2
    
    a2 = 1 / (1 + np.exp(-z2))
    return a2, z2, a1, z1

# 测试前向传播

In [10]:
a2, z2, a1, z1 = forward(x, W1, b1, W2, b2)
eps = 1e-8
a2_clip = np.clip(a2, eps, 1-eps)

loss = -np.mean(y * np.log(a2_clip) + (1 - y) * np.log((1 - a2_clip)))
print(f"损失率:{loss:.6f}")

NameError: name 'forward' is not defined

# 反向传导求梯度

In [46]:
delta2 = a2 - y_train
delta2.shape, a1.shape

((1, 80), (12, 80))

In [47]:
dw2 = np.dot(delta2, a1.T) / len(x)
dw2.shape

(1, 12)

In [48]:
db2 = np.mean(delta2, axis=1, keepdims=True)
db2.shape, b2.shape

((1, 1), (1, 1))

In [50]:
delta1 = np.dot(W2.T, delta2) * (z1 > 0)
delta1.shape

(12, 80)

In [51]:
dw1 = np.dot(delta1, x.T) / len(x)
dw1.shape

(12, 4)

In [52]:
db1 = np.mean(delta1, axis=1, keepdims=True)
db1.shape

(12, 1)

In [54]:
print("dW1 mean:", np.mean(dw1), "std:", np.std(dw1))
print("dW2 mean:", np.mean(dw2), "std:", np.std(dw2))

dW1 mean: -0.0012718988887456461 std: 0.034694264405014656
dW2 mean: -0.0004485740056849943 std: 0.07536674304189536


# 循环训练与更新参数

In [12]:
learning_rate = 0.1  # 这是起始值，可以后续调

# 训练循环（先跑 1000 轮）
for epoch in range(1000):
    # ---- 前向传播 ----
    a2, z1, a1, z2 = forward(x, W1, b1, W2, b2)
    
    # ---- 计算 Loss ----
    eps = 1e-8
    a2_clipped = np.clip(a2, eps, 1 - eps)
    loss = -np.mean(y_train * np.log(a2_clipped) + (1 - y_train) * np.log(1 - a2_clipped))
    
    # ---- 反向传播 ----
    delta2 = a2 - y_train
    dW2 = np.dot(delta2, a1.T) / len(x)
    db2 = np.mean(delta2, axis=1, keepdims=True)
    delta1 = np.dot(W2.T, delta2) * (z1 > 0)
    dW1 = np.dot(delta1, x.T) / len(x)
    db1 = np.mean(delta1, axis=1, keepdims=True)
    
    # ---- 参数更新 ----
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    
    # ---- 打印 ----
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

Epoch 0, Loss: 0.6932
Epoch 100, Loss: 0.0936
Epoch 200, Loss: 0.0935
Epoch 300, Loss: 0.0969
Epoch 400, Loss: 0.0903
Epoch 500, Loss: 0.0864
Epoch 600, Loss: 0.0840
Epoch 700, Loss: 0.0823
Epoch 800, Loss: 0.0549
Epoch 900, Loss: 0.0513


In [ ]:
# 仅改变种子值
## 第一次
Epoch 0, Loss: 0.6932
Epoch 100, Loss: 0.0012
Epoch 200, Loss: 0.0007
Epoch 300, Loss: 0.0006
Epoch 400, Loss: 0.0005
Epoch 500, Loss: 0.0004
Epoch 600, Loss: 0.0004
Epoch 700, Loss: 0.0003
Epoch 800, Loss: 0.0003
Epoch 900, Loss: 0.0003
## 第二次
Epoch 0, Loss: 0.6931
Epoch 100, Loss: 0.0009
Epoch 200, Loss: 0.0005
Epoch 300, Loss: 0.0003
Epoch 400, Loss: 0.0003
Epoch 500, Loss: 0.0002
Epoch 600, Loss: 0.0002
Epoch 700, Loss: 0.0002
Epoch 800, Loss: 0.0002
Epoch 900, Loss: 0.0001
## 第三次
Epoch 0, Loss: 0.6931
Epoch 100, Loss: 0.0034
Epoch 200, Loss: 0.0028
Epoch 300, Loss: 0.0026
Epoch 400, Loss: 0.0025
Epoch 500, Loss: 0.0024
Epoch 600, Loss: 0.0023
Epoch 700, Loss: 0.0022
Epoch 800, Loss: 0.0021
Epoch 900, Loss: 0.0020

# 数据集不进行标准化的过程
## 输出
Epoch 0, Loss: 0.6932
Epoch 100, Loss: 0.0936
Epoch 200, Loss: 0.0935
Epoch 300, Loss: 0.0969
Epoch 400, Loss: 0.0903
Epoch 500, Loss: 0.0864
Epoch 600, Loss: 0.0840
Epoch 700, Loss: 0.0823
Epoch 800, Loss: 0.0549
Epoch 900, Loss: 0.0513

# 测试模型

In [62]:
# ---- 在测试集上评估 ----
a2_test, _, _, _ = forward(x_test, W1, b1, W2, b2)
preds_test = (a2_test > 0.5).astype(int)  # (1, N)
accuracy = np.mean(preds_test == y_test)
print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 1.0000
